# EXP 3 G Sampling Exp

Reliability check for sampling variants using the same preprocessing style as EXP3_C_LogReg_CatBoost copy 2: base, cw, smote, smote+cw, smotenc, smotenc+cw.

Goal: detect whether synthetic patients violate physiological or medical plausibility constraints.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from imblearn.over_sampling import SMOTE, SMOTENC

SEED = 42
TARGET_CANDIDATES = ['hypertension', 'htn', 'target', 'label', 'outcome']
SAMPLING_METHODS = ['base', 'cw', 'smote', 'smotecw', 'smotenc', 'smotencw']

In [ ]:
ROOT = Path.cwd()
DATA_CANDIDATES = [
    ROOT / '0_MISC' / 'merged_clinical_leftjoin.csv',
    ROOT / 'merged_clinical_leftjoin.csv',
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('merged_clinical_leftjoin.csv not found in expected locations')

df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)

def infer_target_column(frame: pd.DataFrame, candidates: list[str]):
    cols_lc = {c.lower(): c for c in frame.columns}
    for c in candidates:
        if c in cols_lc:
            return cols_lc[c]
    return None

target_col = infer_target_column(df, TARGET_CANDIDATES)
if target_col is None:
    sbp = next((c for c in df.columns if c.lower() in {'ave_sbp', 'sbp'}), None)
    dbp = next((c for c in df.columns if c.lower() in {'ave_dbp', 'dbp'}), None)
    if sbp is None or dbp is None:
        raise ValueError('No target found and could not derive from SBP/DBP.')
    df['Hypertension'] = (((pd.to_numeric(df[sbp], errors='coerce') >= 140) | (pd.to_numeric(df[dbp], errors='coerce') >= 90)).fillna(False)).astype(int)
    target_col = 'Hypertension'

df = df.dropna(subset=[target_col]).copy()
y_raw = df[target_col]
if y_raw.nunique() != 2:
    y = pd.Series(LabelEncoder().fit_transform(y_raw.astype(str)), index=y_raw.index, name=target_col)
else:
    y = pd.Series(y_raw.astype(int), index=y_raw.index, name=target_col)

X = df.drop(columns=[target_col]).copy()
print('Target:', target_col, '| Positive rate:', float(y.mean()))

In [ ]:
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

num_imputer = KNNImputer(n_neighbors=5)
cat_imputer = SimpleImputer(strategy='most_frequent')
scaler = StandardScaler()
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

Xtr_num = num_imputer.fit_transform(X_train[num_cols]) if num_cols else np.empty((len(X_train), 0))
Xtr_cat_raw = cat_imputer.fit_transform(X_train[cat_cols]) if cat_cols else np.empty((len(X_train), 0))
Xtr_cat = ord_enc.fit_transform(Xtr_cat_raw) if cat_cols else np.empty((len(X_train), 0))

Xtr_num_scaled = scaler.fit_transform(Xtr_num) if num_cols else np.empty((len(X_train), 0))
Xtr_proc = np.hstack([Xtr_num_scaled, Xtr_cat])
cat_idx = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

print('Train shape:', Xtr_proc.shape, '| numeric:', len(num_cols), '| categorical:', len(cat_cols))

In [ ]:
def run_sampling(method: str):
    if method in {'base', 'cw'}:
        return Xtr_proc.copy(), y_train.to_numpy().copy(), np.array([], dtype=int)

    k_neighbors = max(1, min(5, int((y_train == 1).sum()) - 1, int((y_train == 0).sum()) - 1))

    if method in {'smote', 'smotecw'}:
        X_res, y_res = SMOTE(random_state=SEED, k_neighbors=k_neighbors).fit_resample(Xtr_proc, y_train)
    elif method in {'smotenc', 'smotencw'}:
        if not cat_idx:
            X_res, y_res = SMOTE(random_state=SEED, k_neighbors=k_neighbors).fit_resample(Xtr_proc, y_train)
        else:
            X_res, y_res = SMOTENC(categorical_features=cat_idx, random_state=SEED, k_neighbors=k_neighbors).fit_resample(Xtr_proc, y_train)
    else:
        raise ValueError(f'Unknown method: {method}')

    synth_idx = np.arange(len(Xtr_proc), len(X_res), dtype=int)
    return X_res, np.asarray(y_res), synth_idx

def decode_samples(X_proc: np.ndarray) -> pd.DataFrame:
    n_num = len(num_cols)
    X_num_scaled = X_proc[:, :n_num] if n_num else np.empty((len(X_proc), 0))
    X_cat_ord = X_proc[:, n_num:] if cat_cols else np.empty((len(X_proc), 0))

    X_num = scaler.inverse_transform(X_num_scaled) if n_num else np.empty((len(X_proc), 0))

    decoded = pd.DataFrame(index=np.arange(len(X_proc)))
    for i, c in enumerate(num_cols):
        decoded[c] = X_num[:, i]

    if cat_cols:
        for i, c in enumerate(cat_cols):
            cats = ord_enc.categories_[i]
            idx = np.rint(X_cat_ord[:, i]).astype(int)
            idx = np.clip(idx, -1, len(cats) - 1)
            vals = np.where(idx >= 0, cats[idx], np.nan)
            decoded[c] = vals

    return decoded

In [ ]:
def impossible_flags(df_check: pd.DataFrame) -> pd.DataFrame:
    flags = pd.DataFrame(index=df_check.index)

    def add_numeric_range(col, lo=None, hi=None, strictly_positive=False):
        if col not in df_check.columns:
            return
        s = pd.to_numeric(df_check[col], errors='coerce')
        bad = pd.Series(False, index=df_check.index)
        if lo is not None:
            bad |= s < lo
        if hi is not None:
            bad |= s > hi
        if strictly_positive:
            bad |= s <= 0
        flags[f'{col}_impossible'] = bad.fillna(False)

    add_numeric_range('age', lo=0, hi=120)
    add_numeric_range('weight', lo=20, hi=350, strictly_positive=True)
    add_numeric_range('height', lo=80, hi=260, strictly_positive=True)
    add_numeric_range('waist', lo=30, hi=250, strictly_positive=True)
    add_numeric_range('hip', lo=30, hi=250, strictly_positive=True)

    for c in [x for x in df_check.columns if x.lower().startswith('fg') or x.lower().startswith('epwt_fg') or x.startswith('Total_')]:
        s = pd.to_numeric(df_check[c], errors='coerce')
        flags[f'{c}_negative'] = (s < 0).fillna(False)

    if 'sex' in df_check.columns:
        sx = pd.to_numeric(df_check['sex'], errors='coerce')
        flags['sex_code_invalid'] = (~sx.isin([1, 2])).fillna(False)

    if {'smoke_status', 'current_smoking', 'ever_smk'}.issubset(df_check.columns):
        ss = pd.to_numeric(df_check['smoke_status'], errors='coerce')
        cs = pd.to_numeric(df_check['current_smoking'], errors='coerce')
        es = pd.to_numeric(df_check['ever_smk'], errors='coerce')
        smoke_bad = ((ss == 0) & ((cs != 0) | (es != 0))) | ((ss == 1) & (~cs.isin([1, 2, 3]))) | ((ss == 2) & (cs != 0))
        flags['smoking_cascade_invalid'] = smoke_bad.fillna(False)

    if {'alcohol_status', 'alcohol', 'con_alcohol', 'drnk_30days'}.issubset(df_check.columns):
        ast = pd.to_numeric(df_check['alcohol_status'], errors='coerce')
        alc = pd.to_numeric(df_check['alcohol'], errors='coerce')
        con = pd.to_numeric(df_check['con_alcohol'], errors='coerce')
        d30 = pd.to_numeric(df_check['drnk_30days'], errors='coerce')
        alcohol_bad = ((ast == 0) & (alc != 0)) | ((ast == 1) & ((alc != 1) | (con != 1) | (~d30.isin([0, 1])))) | ((ast == 2) & ((alc != 1) | (con != 0)))
        flags['alcohol_cascade_invalid'] = alcohol_bad.fillna(False)

    return flags

summary_rows = []
violation_details = {}

for method in SAMPLING_METHODS:
    X_res, y_res, synth_idx = run_sampling(method)
    if len(synth_idx) == 0:
        summary_rows.append({
            'method': method,
            'class_weight_active': method in {'cw', 'smotecw', 'smotencw'},
            'n_synthetic': 0,
            'n_impossible': 0,
            'impossible_rate': 0.0,
            'note': 'No synthetic rows generated'
        })
        continue

    syn_df = decode_samples(X_res[synth_idx])
    flags = impossible_flags(syn_df)
    impossible_any = flags.any(axis=1) if not flags.empty else pd.Series(False, index=syn_df.index)

    summary_rows.append({
        'method': method,
        'class_weight_active': method in {'cw', 'smotecw', 'smotencw'},
        'n_synthetic': int(len(syn_df)),
        'n_impossible': int(impossible_any.sum()),
        'impossible_rate': float(impossible_any.mean()),
        'note': ''
    })

    violation_details[method] = (flags.mean().sort_values(ascending=False).head(12) * 100.0).rename('percent').to_frame()

sampling_reliability = pd.DataFrame(summary_rows).sort_values(['impossible_rate', 'n_impossible'])
sampling_reliability

In [ ]:
# Inspect dominant violation sources for a chosen method
method_to_inspect = 'smotencw'
violation_details.get(method_to_inspect, pd.DataFrame({'percent': []}))